# 📊 통신사 고객 이탈 데이터(churn.csv) 탐색적 데이터 분석 (EDA)

본 노트북은 통신사 고객 데이터를 기반으로 고객이 서비스를 해지하는 요인(이탈 요인, Churn)을 분석하기 위해 설계되었습니다.
기술 명세서에 제시된 6단계 분석 흐름에 따라 진행됩니다.

## 🛠️ 1. 라이브러리 및 데이터 로드

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# 시각화 스타일 설정
sns.set_theme(style="whitegrid")
try:
    plt.rcParams['font.family'] = 'Malgun Gothic'  # Windows 환경에서 한글 깨짐 방지
except:
    pass
plt.rcParams['axes.unicode_minus'] = False

# 데이터 로드 (로컬에 파일이 없으면 깃허브에서 자동 다운로드)
if not os.path.exists('churn.csv'):
    print('churn.csv 파일 다운로드 중...')
    !wget -q https://raw.githubusercontent.com/diffth/AIBook/main/VibeCodingAcorn/vibeCoding-day06/churn.csv
df = pd.read_csv('churn.csv')
print(f"데이터 크기: {df.shape}")

## 🔍 1. 데이터 프로파일링 및 무결성 검증
데이터의 전반적인 구조, 변수 타입, 결측치 및 중복값 여부를 파악합니다. 특히 `Churn` 변수 값에 있는 온점(`.`)을 깔끔하게 처리하는 전처리를 수행합니다.

In [ ]:
# 상위 5개 데이터 확인
print("--- 상위 5행 데이터 ---")
display(df.head())

# 데이터 요약 정보 확인
print("\n--- 데이터 기본 정보 ---")
df.info()

# 결측치 및 중복값 검사
print(f"\n결측치 개수:\n{df.isnull().sum()}")
print(f"\n중복된 데이터 수: {df.duplicated().sum()}")

# 수치형 변수 요약 통계
print("\n--- 수치형 피처 요약 통계량 ---")
display(df.describe())

# [전처리] Churn 컬럼 값 정화 (온점 제거 및 불리언/카테고리 변환)
df['Churn'] = df['Churn'].astype(str).str.replace('.', '', regex=False).str.strip()
# True/False 불리언 형태로 명확화
df['Churn'] = df['Churn'].map({'True': True, 'False': False})
print(f"\nChurn 고유값 정화 완료: {df['Churn'].unique()}")

## 🎯 2. 타겟 변수(이탈 여부: Churn) 분포 분석
이탈 고객과 유지 고객의 비율을 확인하여 데이터의 불균형(Class Imbalance) 정도를 분석합니다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 카운트 플롯
sns.countplot(x='Churn', data=df, ax=axes[0], palette='pastel')
axes[0].set_title('고객 이탈 여부 빈도 (Churn Count)')
axes[0].set_xlabel('이탈 여부')
axes[0].set_ylabel('고객 수')

# 파이 차트
churn_counts = df['Churn'].value_counts()
axes[1].pie(churn_counts, labels=['유지 (False)', '이탈 (True)'], autopct='%1.1f%%', startangle=90, colors=['lightblue', 'lightcoral'])
axes[1].set_title('고객 이탈 비율 (Churn Ratio)')

plt.tight_layout()
plt.show()

print(f"상세 분포:\n{churn_counts}")

## 📞 3. 가입 요금제 유형(Intl_Plan, Vmail_Plan)에 따른 고객 이탈 패턴 분석
국제 통화 요금제(`Intl_Plan`)와 음성 사서함 요금제(`Vmail_Plan`) 가입 여부에 따른 이탈율 변화를 파악합니다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 1. Intl_Plan vs Churn
intl_churn = pd.crosstab(df['Intl_Plan'], df['Churn'], normalize='index') * 100
intl_churn.plot(kind='bar', stacked=True, ax=axes[0], color=['skyblue', 'salmon'])
axes[0].set_title('국제 요금제 가입 여부별 이탈 비율')
axes[0].set_xlabel('국제 요금제 가입 여부 (Intl_Plan)')
axes[0].set_ylabel('비율 (%)')
axes[0].legend(['유지', '이탈'])

# 2. Vmail_Plan vs Churn
vmail_churn = pd.crosstab(df['Vmail_Plan'], df['Churn'], normalize='index') * 100
vmail_churn.plot(kind='bar', stacked=True, ax=axes[1], color=['skyblue', 'salmon'])
axes[1].set_title('음성사서함 요금제 가입 여부별 이탈 비율')
axes[1].set_xlabel('음성사서함 요금제 가입 여부 (Vmail_Plan)')
axes[1].set_ylabel('비율 (%)')
axes[1].legend(['유지', '이탈'])

plt.tight_layout()
plt.show()

print("--- 국제 요금제별 이탈 테이블 (%): ---")
print(intl_churn)
print("\n--- 음성사서함 요금제별 이탈 테이블 (%): ---")
print(vmail_churn)

## 📊 4. 시간대별 통화 사용량(Mins/Calls/Charge) 및 요금 상관관계 분석
통화 시간(Mins), 통화 횟수(Calls), 통화 요금(Charge) 간의 관계와 시간대별 특성을 분석합니다. 상관관계 히트맵을 그려서 변수 간 다중공선성을 검토하고, 파생 피처를 생성합니다.

In [ ]:
# 수치형 변수들 선택
numerical_cols = [
    'Account_Length', 'Vmail_Message', 
    'Day_Mins', 'Day_Calls', 'Day_Charge',
    'Eve_Mins', 'Eve_Calls', 'Eve_Charge',
    'Night_Mins', 'Night_Calls', 'Night_Charge',
    'Intl_Mins', 'Intl_Calls', 'Intl_Charge',
    'CustServ_Calls'
]

# 상관관계 매트릭스 계산
corr_matrix = df[numerical_cols].corr()

# 히트맵 시각화
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', square=True, cbar_kws={'shrink': .8})
plt.title('수치형 피처 간 상관관계 히트맵 (Correlation Heatmap)')
plt.show()

# [피처 엔지니어링] 파생 변수 생성 (전체 통화량 및 요금 합산)
df['Total_Mins'] = df['Day_Mins'] + df['Eve_Mins'] + df['Night_Mins']
df['Total_Charge'] = df['Day_Charge'] + df['Eve_Charge'] + df['Night_Charge']

plt.figure(figsize=(10, 5))
sns.boxplot(x='Churn', y='Total_Charge', data=df, palette='Set2')
plt.title('고객 이탈 여부에 따른 총 통화 요금 분포 (Total Charge vs Churn)')
plt.xlabel('이탈 여부')
plt.ylabel('총 통화 요금 (Total Charge)')
plt.show()

## 🏢 5. 고객 서비스 센터 통화 횟수(CustServ_Calls)와 이탈률 관계 분석
고객 센터 문의 횟수(`CustServ_Calls`)가 늘어남에 따라 이탈률이 급격하게 치솟는 위험 임계점(Threshold)을 규명합니다.

In [ ]:
plt.figure(figsize=(12, 6))

# 고객 서비스 센터 통화 횟수별 이탈률 산출
cust_service_churn = df.groupby('CustServ_Calls')['Churn'].mean() * 100

# 라인 차트 시각화
sns.lineplot(x=cust_service_churn.index, y=cust_service_churn.values, marker='o', color='crimson', linewidth=2.5)
plt.title('고객 서비스 센터 통화 횟수별 이탈률 변화 (Churn Rate by CustServ_Calls)')
plt.xlabel('고객 서비스 센터 통화 횟수 (CustServ_Calls)')
plt.ylabel('이탈률 (%)')
plt.axhline(y=df['Churn'].mean()*100, color='grey', linestyle='--', label='전체 평균 이탈률')
plt.axvline(x=4, color='orange', linestyle=':', label='위험 임계점 (4회)')
plt.legend()
plt.xticks(cust_service_churn.index)
plt.show()

print("--- 고객 서비스 통화 횟수별 이탈률 (%) ---")
print(cust_service_churn.round(2))

## 💡 6. 데이터 전처리 가이드라인 및 모델링 준비 계획

### 📊 분석 요약
1. **이탈 불균형**: 데이터 셋 내 고객 이탈율은 소수 클래스로서 불균형 상태입니다.
2. **국제 요금제 가입자**: `Intl_Plan` 가입자 중 이탈하는 고객의 비율이 현저히 높게 관측되었습니다.
3. **고객 서비스 센터**: `CustServ_Calls`가 4회 이상인 고객군에서 이탈률이 50% 이상으로 급상승합니다.
4. **통화 요금**: 총 통화 요금이 높을수록 이탈 경향이 강해집니다.

### ⚙️ 머신러닝 연계를 위한 데이터 전처리 가이드라인
1. **불필요한 변수 제거**: `Phone` 변수와 같이 고유값 식별자는 분석 및 예측 성능에 노이즈이므로 드롭합니다.
2. **인코딩(Encoding)**:
   * `State` 등 고다차원 범주형 피처: 타겟 인코딩 또는 원핫 인코딩 적용
   * `Intl_Plan`, `Vmail_Plan`: 0 또는 1의 이진 인코딩(Label Encoding) 적용
3. **스케일링(Scaling)**: 통화 요금 및 사용 시간 변수 등의 편차를 줄이기 위해 `StandardScaler` 또는 `RobustScaler`를 활용합니다.
4. **불균형 해소**: 머신러닝 분류기 학습 시 `SMOTE` 오버샘플링 기법 또는 모델 파라미터의 `class_weight='balanced'` 설정을 추천합니다.